# Prévision de pluie en Australie — 1/3 · Exploration (EDA)

Dataset Kaggle *Rain in Australia*. Cible : `RainTomorrow` (pluie demain).

**Cette partie** : chargement, qualité des données, variable cible & déséquilibre, baselines, corrélations, dimensions géographique & saisonnière.

Suite : `02_preprocessing.ipynb` (préparation) puis `03_modelisation.ipynb` (modèles).

## 0. Setup & chargement des données

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
RANDOM_STATE = 42

In [ ]:
# Résolution robuste du chemin des données : fonctionne que le notebook soit
# lancé depuis la racine du projet ou depuis le dossier notebooks/.
CANDIDATES = [
    Path("Data/weatherAUS.csv"),
    Path("../Data/weatherAUS.csv"),
    Path.home() / "Desktop/MlOps_Meteo-Liora/Data/weatherAUS.csv",
]
DATA_PATH = next((p for p in CANDIDATES if p.exists()), None)
assert DATA_PATH is not None, "weatherAUS.csv introuvable (vérifier le dossier Data/)"

df = pd.read_csv(DATA_PATH, na_values=["NA"])
print(f"Chargé depuis : {DATA_PATH}")
print(f"Dimensions    : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
df.head()

## 1. Exploration des données (EDA)

On caractérise la structure, la qualité (valeurs manquantes), la variable cible et ses liens
avec les variables explicatives, puis les dimensions géographique et saisonnière.

In [ ]:
# Types, mémoire et doublons
df.info()
print("\nDoublons exacts :", df.duplicated().sum())

### 1.1 Valeurs manquantes

In [ ]:
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(missing_pct.index[::-1], missing_pct.values[::-1], color="#4C72B0")
ax.set_xlabel("% de valeurs manquantes")
ax.set_title("Valeurs manquantes par colonne")
for i, v in enumerate(missing_pct.values[::-1]):
    ax.text(v + 0.4, i, f"{v:.1f}%", va="center", fontsize=8)
plt.tight_layout()
plt.show()
missing_pct.round(2)

**Lecture.** Quatre colonnes sont très lacunaires : `Sunshine` (~48 %), `Evaporation` (~43 %),
`Cloud3pm` (~41 %), `Cloud9am` (~38 %). Les autres restent sous ~10 %. La cible `RainTomorrow`
est manquante sur ~2,2 % des lignes (ces lignes seront retirées). Plutôt que de **supprimer**
`Sunshine`/`Cloud` (qui comptent parmi les meilleurs prédicteurs, cf. §1.3), on les **conservera
et imputera** — au prix d'une incertitude assumée.

### 1.2 Variable cible & déséquilibre

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
order = ["No", "Yes"]
sns.countplot(x="RainTomorrow", data=df, order=order,
              hue="RainTomorrow", hue_order=order, palette="viridis", legend=False, ax=ax)
ax.set_title("Distribution de RainTomorrow")
ax.set_xlabel("Pluie demain ?")
ax.set_ylabel("Nombre de jours")
plt.tight_layout()
plt.show()

vc = df["RainTomorrow"].value_counts(dropna=False)
base_rate = 100 * vc.get("Yes", 0) / (vc.get("Yes", 0) + vc.get("No", 0))
print(vc)
print(f"\nTaux de base (Yes) = {base_rate:.2f}%  ->  classe fortement déséquilibrée (~78/22).")

**Lecture.** Il pleut le lendemain dans environ **22 %** des cas. Le déséquilibre (~78/22) est
important : l'*accuracy* seule sera trompeuse (un modèle disant toujours « non » atteint déjà ~78 %).
On suivra donc surtout le **recall/F1 de la classe « pluie »**.

### 1.3 Baselines de référence

Avant tout modèle, on fixe deux repères simples — tout modèle « utile » doit les battre.

In [ ]:
valid = df.dropna(subset=["RainTomorrow"]).copy()

# Baseline 1 : prédire toujours "No"
acc_always_no = 100 * (valid["RainTomorrow"] == "No").mean()

# Baseline 2 : persistance -> "demain = aujourd'hui" (RainTomorrow == RainToday)
persist = valid.dropna(subset=["RainToday"])
acc_persist = 100 * (persist["RainTomorrow"] == persist["RainToday"]).mean()
p_yes_if_today_yes = 100 * (persist.loc[persist.RainToday == "Yes", "RainTomorrow"] == "Yes").mean()
p_yes_if_today_no = 100 * (persist.loc[persist.RainToday == "No", "RainTomorrow"] == "Yes").mean()

print(f"Baseline 'toujours Non'        : accuracy = {acc_always_no:.2f}%")
print(f"Baseline persistance (=RainToday): accuracy = {acc_persist:.2f}%")
print(f"  P(pluie demain | il a plu aujourd'hui)   = {p_yes_if_today_yes:.1f}%")
print(f"  P(pluie demain | pas de pluie aujourd'hui) = {p_yes_if_today_no:.1f}%")

**Lecture.** La barre à battre est ~**78 %** (toujours-Non) ou ~**76 %** (persistance). La pluie
de la veille triple la probabilité de pluie le lendemain (~46 % vs ~15 %) : `RainToday` est informatif
(et, comme `Rainfall`, à surveiller pour le risque de **fuite** si mal utilisé).

### 1.4 Variables explicatives — corrélations avec la cible

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()
y_bin = df["RainTomorrow"].map({"Yes": 1, "No": 0})

corr = {}
for c in num_cols:
    sub = pd.DataFrame({"x": df[c], "y": y_bin}).dropna()
    corr[c] = sub["x"].corr(sub["y"])
corr = pd.Series(corr).sort_values(key=np.abs, ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
colors = ["#C44E52" if v > 0 else "#4C72B0" for v in corr.values]
ax.barh(corr.index, corr.values, color=colors)
ax.axvline(0, color="k", lw=0.8)
ax.set_title("Corrélation point-bisériale avec RainTomorrow")
ax.set_xlabel("corrélation (rouge = positive, bleu = négative)")
plt.tight_layout()
plt.show()
corr.sort_values(key=np.abs, ascending=False).round(3)

**Lecture.** Les prédicteurs les plus forts sont l'**ensoleillement** `Sunshine` (≈ −0,45),
l'**humidité à 15 h** `Humidity3pm` (≈ +0,45), la **nébulosité** `Cloud3pm` (≈ +0,38) et la
**pression** (≈ −0,23). Physiquement cohérent : moins de soleil + plus d'humidité/nuages + pression
basse ⇒ pluie plus probable.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, col in zip(axes, ["Humidity3pm", "Sunshine", "Pressure3pm"]):
    sns.boxplot(x="RainTomorrow", y=col, data=df, order=["No", "Yes"],
                hue="RainTomorrow", hue_order=["No", "Yes"],
                palette="coolwarm", legend=False, ax=ax)
    ax.set_title(f"{col} selon RainTomorrow")
    ax.set_xlabel("Pluie demain ?")
plt.tight_layout()
plt.show()

### 1.5 Dimensions géographique & saisonnière

In [ ]:
rate_loc = (valid.assign(y=lambda d: (d.RainTomorrow == "Yes"))
            .groupby("Location")["y"].mean().mul(100).sort_values())
top = pd.concat([rate_loc.head(8), rate_loc.tail(8)])

df["_month"] = pd.to_datetime(df["Date"], errors="coerce").dt.month
rate_month = (df.dropna(subset=["RainTomorrow", "_month"])
              .assign(y=lambda d: (d.RainTomorrow == "Yes"))
              .groupby("_month")["y"].mean().mul(100))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].barh(top.index, top.values, color="#55A868")
axes[0].set_title("Taux de pluie demain — 8 stations + sèches / + pluvieuses")
axes[0].set_xlabel("% de jours suivis de pluie")
axes[1].bar(rate_month.index, rate_month.values, color="#4C72B0")
axes[1].set_title("Saisonnalité — taux de pluie demain par mois")
axes[1].set_xlabel("mois")
axes[1].set_ylabel("% de jours suivis de pluie")
axes[1].set_xticks(range(1, 13))
plt.tight_layout()
plt.show()
print("Mois le + pluvieux:", int(rate_month.idxmax()), f"({rate_month.max():.1f}%)",
      "| + sec:", int(rate_month.idxmin()), f"({rate_month.min():.1f}%)")

**Lecture.** Forte hétérogénéité géographique : de ~7 % (`Woomera`, `Uluru`, désert intérieur)
à ~37 % (`Portland`, côte sud humide). Saisonnalité marquée : pic en **hiver austral** (juin–août,
~26–27 %), creux en été (janvier, ~19 %). La station et le mois sont donc des features pertinentes.